# PySpark & Databricks Interview Preparation

This notebook covers the essential concepts you need to master for a Databricks freelance interview.

**Topics Covered:**
1. PySpark Fundamentals (Quick Refresher)
2. Delta Lake (Non-Negotiable)
3. Medallion Architecture (Bronze → Silver → Gold)
4. Structured Streaming & Auto Loader
5. Performance Optimization
6. Unity Catalog
7. Data Quality
8. Common Interview Scenarios

---

## Setup

### Running in Databricks (Recommended)
Just run the cells - `spark` is pre-initialized. All features work.

### Running Locally
PySpark requires **Java 8, 11, or 17** installed. Check with `java -version`.

**Install Java (if needed):**
```bash
# Ubuntu/Debian
sudo apt install openjdk-17-jdk

# macOS
brew install openjdk@17

# Windows - download from https://adoptium.net/
```

**Then set JAVA_HOME:**
```bash
export JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64  # Linux
export JAVA_HOME=$(/usr/libexec/java_home -v 17)     # macOS
```

**Note:** Delta Lake and Databricks-specific features (Auto Loader, Unity Catalog, DLT) require Databricks. These cells are marked with `# DATABRICKS ONLY`.

In [2]:
# Standard imports you'll use in every Databricks project
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

# Initialize SparkSession
# In Databricks: 'spark' is pre-initialized, this cell is a no-op
# Locally: Creates a new SparkSession (requires Java installed)

try:
    # Check if spark already exists (Databricks environment)
    spark
    print("Using existing Databricks SparkSession")
except NameError:
    # Create local SparkSession
    spark = SparkSession.builder \
        .appName("InterviewPrep") \
        .master("local[*]") \
        .config("spark.driver.memory", "4g") \
        .getOrCreate()
    print(f"Created local SparkSession (Spark version: {spark.version})")

# Verify spark is working
spark.sql("SELECT 'PySpark is ready!' as status").show(truncate=False)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/19 21:41:25 WARN Utils: Your hostname, Michas-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.178.25 instead (on interface en0)
26/02/19 21:41:25 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/19 21:41:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Created local SparkSession (Spark version: 4.1.1)
+-----------------+
|status           |
+-----------------+
|PySpark is ready!|
+-----------------+



---
# 1. PySpark Fundamentals (Quick Refresher)

Before diving into Databricks-specific features, ensure you're comfortable with core PySpark operations.

## 1.1 Creating DataFrames

In [3]:
# Method 1: From a list of tuples
data = [
    ("M001", "2025-01-15 10:00:00", 150.5),
    ("M001", "2025-01-15 11:00:00", 152.3),
    ("M002", "2025-01-15 10:00:00", 200.0),
    ("M002", "2025-01-15 11:00:00", None),  # Intentional null for demo
]

schema = StructType([
    StructField("meter_id", StringType(), False),
    StructField("reading_timestamp", StringType(), True),
    StructField("reading_value", DoubleType(), True)
])

df = spark.createDataFrame(data, schema)
df.show()

+--------+-------------------+-------------+
|meter_id|  reading_timestamp|reading_value|
+--------+-------------------+-------------+
|    M001|2025-01-15 10:00:00|        150.5|
|    M001|2025-01-15 11:00:00|        152.3|
|    M002|2025-01-15 10:00:00|        200.0|
|    M002|2025-01-15 11:00:00|         NULL|
+--------+-------------------+-------------+



In [ ]:
# Method 2: From a dictionary (useful for quick testing)
from pyspark.sql import Row

data_dict = [
    Row(customer_id="C001", name="Alice", region="North"),
    Row(customer_id="C002", name="Bob", region="South"),
]

df_customers = spark.createDataFrame(data_dict)
df_customers.show()

## 1.2 Essential DataFrame Operations

In [ ]:
# SELECT columns
df.select("meter_id", "reading_value").show()

# SELECT with expressions
df.select(
    F.col("meter_id"),
    F.col("reading_value"),
    (F.col("reading_value") * 1.1).alias("reading_plus_10_percent")
).show()

In [ ]:
# FILTER / WHERE (interchangeable)
df.filter(F.col("reading_value").isNotNull()).show()
df.where(F.col("reading_value") > 151).show()

In [ ]:
# Add/Modify columns with withColumn
df_transformed = df \
    .withColumn("reading_timestamp", F.to_timestamp("reading_timestamp")) \
    .withColumn("reading_date", F.to_date("reading_timestamp")) \
    .withColumn("is_valid", F.col("reading_value").isNotNull())

df_transformed.show()
df_transformed.printSchema()

In [ ]:
# GROUP BY and Aggregations
df.groupBy("meter_id").agg(
    F.count("*").alias("reading_count"),
    F.sum("reading_value").alias("total_reading"),
    F.avg("reading_value").alias("avg_reading"),
    F.min("reading_value").alias("min_reading"),
    F.max("reading_value").alias("max_reading")
).show()

In [ ]:
# ORDER BY
df.orderBy(F.col("reading_value").desc()).show()
df.orderBy(F.col("meter_id").asc(), F.col("reading_timestamp").desc()).show()

## 1.3 Joins

In [ ]:
# Create sample DataFrames for joins
meters = spark.createDataFrame([
    ("M001", "C001", "Residential"),
    ("M002", "C002", "Commercial"),
    ("M003", "C003", "Industrial"),  # No readings for this meter
], ["meter_id", "customer_id", "meter_type"])

readings = spark.createDataFrame([
    ("M001", 150.5),
    ("M001", 152.3),
    ("M002", 200.0),
    ("M004", 50.0),  # Unknown meter
], ["meter_id", "reading_value"])

In [ ]:
# INNER JOIN (default) - only matching rows
meters.join(readings, "meter_id", "inner").show()

In [ ]:
# LEFT JOIN - all from left, matching from right
meters.join(readings, "meter_id", "left").show()

In [ ]:
# FULL OUTER JOIN - all from both sides
meters.join(readings, "meter_id", "full").show()

In [ ]:
# Join with different column names
df_a = spark.createDataFrame([("1", "Alice")], ["id", "name"])
df_b = spark.createDataFrame([("1", 100)], ["customer_id", "amount"])

df_a.join(df_b, df_a.id == df_b.customer_id, "inner").show()

## 1.4 Window Functions

**Critical for interviews!** Window functions are used constantly in real projects.

In [ ]:
# Sample data for window functions
sales_data = spark.createDataFrame([
    ("North", "2025-01-01", 100),
    ("North", "2025-01-02", 150),
    ("North", "2025-01-03", 120),
    ("South", "2025-01-01", 200),
    ("South", "2025-01-02", 180),
    ("South", "2025-01-03", 220),
], ["region", "date", "sales"])

sales_data.show()

In [ ]:
# ROW_NUMBER - assign unique row numbers within each partition
window_spec = Window.partitionBy("region").orderBy(F.col("sales").desc())

sales_data.withColumn("rank_in_region", F.row_number().over(window_spec)).show()

In [ ]:
# Running total / cumulative sum
window_running = Window.partitionBy("region").orderBy("date").rowsBetween(Window.unboundedPreceding, Window.currentRow)

sales_data.withColumn("cumulative_sales", F.sum("sales").over(window_running)).show()

In [ ]:
# LAG / LEAD - access previous/next row values
window_ordered = Window.partitionBy("region").orderBy("date")

sales_data \
    .withColumn("prev_day_sales", F.lag("sales", 1).over(window_ordered)) \
    .withColumn("next_day_sales", F.lead("sales", 1).over(window_ordered)) \
    .withColumn("day_over_day_change", F.col("sales") - F.col("prev_day_sales")) \
    .show()

---
# 2. Delta Lake (Non-Negotiable)

**Every Databricks project uses Delta Lake.** You must know this cold.

## What is Delta Lake?
- Open-source storage layer bringing **ACID transactions** to data lakes
- Stores data as **Parquet files** + a **transaction log** (`_delta_log/`)
- Enables: time travel, schema enforcement, concurrent reads/writes

## 2.1 Basic Delta Operations

In [ ]:
# Write as Delta (default in Databricks)
df.write.format("delta").mode("overwrite").saveAsTable("bronze.meter_readings")

# Alternative: save to path
df.write.format("delta").mode("overwrite").save("/delta/meter_readings")

In [ ]:
# Read Delta
df_delta = spark.read.table("bronze.meter_readings")

# Or from path
df_delta = spark.read.format("delta").load("/delta/meter_readings")

## 2.2 Time Travel

Query previous versions of your data — incredibly useful for debugging and auditing.

In [ ]:
# Query by version number
df_version_5 = spark.read.option("versionAsOf", 5).table("bronze.meter_readings")

# Query by timestamp
df_yesterday = spark.read.option("timestampAsOf", "2025-01-15").table("bronze.meter_readings")

# View table history
spark.sql("DESCRIBE HISTORY bronze.meter_readings").show(truncate=False)

In [ ]:
# RESTORE a table to a previous version (rollback a bad write)
spark.sql("RESTORE TABLE bronze.meter_readings TO VERSION AS OF 5")

## 2.3 MERGE (Upsert)

**This comes up in EVERY interview.** MERGE is how you handle incremental updates.

In [ ]:
from delta.tables import DeltaTable

# Assume we have a target table and new incoming data
# Target: existing customer records
# Source: new/updated customer records

target = DeltaTable.forName(spark, "silver.customers")
# Or: target = DeltaTable.forPath(spark, "/delta/customers")

source_df = spark.createDataFrame([
    ("C001", "Alice Updated", "alice@new.com"),
    ("C003", "Charlie", "charlie@example.com"),  # New customer
], ["customer_id", "name", "email"])

# MERGE: Update existing, Insert new
target.alias("t").merge(
    source_df.alias("s"),
    "t.customer_id = s.customer_id"  # Match condition
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

In [ ]:
# More granular MERGE control
target.alias("t").merge(
    source_df.alias("s"),
    "t.customer_id = s.customer_id"
).whenMatchedUpdate(
    condition="s.name != t.name",  # Only update if name changed
    set={"name": "s.name", "email": "s.email", "updated_at": "current_timestamp()"}
).whenNotMatchedInsert(
    values={
        "customer_id": "s.customer_id",
        "name": "s.name",
        "email": "s.email",
        "created_at": "current_timestamp()",
        "updated_at": "current_timestamp()"
    }
).execute()

## 2.4 Schema Evolution

Handle schema changes gracefully.

In [ ]:
# Schema enforcement (default) - will FAIL if schema doesn't match
# This protects your data quality

# Schema evolution - allow new columns to be added
df_with_new_column = df.withColumn("new_field", F.lit("value"))

df_with_new_column.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("bronze.meter_readings")

## 2.5 Key Interview Questions on Delta Lake

| Question | Answer |
|----------|--------|
| How would you handle late-arriving data? | MERGE/upsert |
| How do you roll back a bad write? | `RESTORE TABLE ... TO VERSION AS OF n` |
| Overwrite vs Merge? | Overwrite = full replace; Merge = incremental |
| Schema changes? | Schema enforcement (default) vs. schema evolution (`mergeSchema`) |

---
# 3. Medallion Architecture (Bronze → Silver → Gold)

**The standard Databricks data engineering pattern.** You must be able to whiteboard this.

| Layer | Purpose | Characteristics |
|-------|---------|----------------|
| **Bronze** | Raw ingestion | Append-only, minimal transformation, keep source as-is |
| **Silver** | Cleaned data | Deduplicated, typed, joined, business logic applied |
| **Gold** | Business-ready | Aggregated, optimized for reporting/ML, often star-schema |

In [ ]:
# ============================================
# BRONZE LAYER: Raw ingestion
# ============================================

# Read raw data (CSV, JSON, Parquet, etc.)
raw_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/raw/meter_readings/")

# Add metadata columns for auditing
bronze_df = raw_df \
    .withColumn("_ingested_at", F.current_timestamp()) \
    .withColumn("_source_file", F.input_file_name())

# Write to Bronze - APPEND mode to preserve history
bronze_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("bronze.meter_readings")

In [ ]:
# ============================================
# SILVER LAYER: Cleaned and transformed
# ============================================

# Read from Bronze
bronze_df = spark.read.table("bronze.meter_readings")

# Clean and transform
silver_df = bronze_df \
    .filter(F.col("reading_value").isNotNull()) \
    .filter(F.col("reading_value") > 0) \
    .withColumn("reading_value", F.col("reading_value").cast("double")) \
    .withColumn("reading_timestamp", F.to_timestamp("reading_timestamp")) \
    .dropDuplicates(["meter_id", "reading_timestamp"])

# Write to Silver
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.meter_readings")

In [ ]:
# ============================================
# GOLD LAYER: Aggregated for business use
# ============================================

# Read from Silver
silver_df = spark.read.table("silver.meter_readings")

# Aggregate for reporting
gold_df = silver_df \
    .groupBy(
        "meter_id",
        F.date_trunc("month", "reading_timestamp").alias("month")
    ) \
    .agg(
        F.sum("reading_value").alias("total_consumption"),
        F.count("*").alias("reading_count"),
        F.avg("reading_value").alias("avg_reading"),
        F.min("reading_value").alias("min_reading"),
        F.max("reading_value").alias("max_reading")
    )

# Write to Gold
gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.monthly_consumption")

---
# 4. Structured Streaming & Auto Loader

Most Databricks projects involve incremental/streaming processing.

## 4.1 Auto Loader

**THE standard way to incrementally ingest files in Databricks.**

In [ ]:
# Auto Loader - incrementally process new files as they arrive
df_stream = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("cloudFiles.schemaLocation", "/checkpoints/schema/meter_readings") \
    .option("header", "true") \
    .load("/raw/meter_readings/")

# Write stream to Delta
df_stream.writeStream \
    .format("delta") \
    .option("checkpointLocation", "/checkpoints/meter_readings") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .toTable("bronze.meter_readings")

## 4.2 Key Streaming Concepts

| Concept | Explanation |
|---------|-------------|
| **Checkpointing** | Spark tracks what files/offsets already processed. Enables exactly-once semantics. |
| **Trigger Modes** | `availableNow=True` (batch-like), `processingTime="5 minutes"` (micro-batch), `continuous` (rare) |
| **Auto Loader vs COPY INTO** | Auto Loader scales better, tracks state, handles schema evolution. COPY INTO is simpler but idempotent-only. |

In [ ]:
# Different trigger modes

# Process all available data, then stop (like a batch job)
.trigger(availableNow=True)

# Process in micro-batches every 5 minutes
.trigger(processingTime="5 minutes")

# Process once, then stop
.trigger(once=True)

In [ ]:
# COPY INTO - simpler alternative (SQL-based)
spark.sql("""
    COPY INTO bronze.meter_readings
    FROM '/raw/meter_readings/'
    FILEFORMAT = CSV
    FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true')
    COPY_OPTIONS ('mergeSchema' = 'true')
""")

---
# 5. Performance Optimization

**This separates junior from senior freelancers.** Interviewers will probe here.

## 5.1 Partitioning

In [ ]:
# GOOD: Partition by low-cardinality columns (year, month, region)
df.write \
    .partitionBy("year", "month") \
    .format("delta") \
    .saveAsTable("silver.readings")

# BAD: Partitioning by high-cardinality column → millions of tiny files!
# df.write.partitionBy("customer_id")  # DON'T DO THIS

## 5.2 Z-Ordering (Delta-specific)

In [ ]:
# Z-Order colocates related data in the same files for faster filtering
# Run this periodically as maintenance

spark.sql("OPTIMIZE silver.readings ZORDER BY (meter_id)")

# Multiple columns
spark.sql("OPTIMIZE silver.readings ZORDER BY (meter_id, reading_date)")

## 5.3 Liquid Clustering (Modern Approach)

In [ ]:
# Liquid Clustering replaces Z-Order + partitioning
# No more manual OPTIMIZE needed - automatic!

df.write \
    .format("delta") \
    .clusterBy("meter_id", "reading_date") \
    .saveAsTable("silver.readings")

# Or via SQL
spark.sql("""
    CREATE TABLE silver.readings
    CLUSTER BY (meter_id, reading_date)
    AS SELECT * FROM bronze.readings
""")

## 5.4 Broadcast Joins

In [ ]:
# Broadcast small dimension tables to avoid expensive shuffle
# Use when one DataFrame is small (< 10MB by default)

df_small = spark.read.table("dim.regions")  # Small lookup table
df_big = spark.read.table("fact.sales")  # Large fact table

# Explicit broadcast hint
df_joined = df_big.join(F.broadcast(df_small), "region_id", "left")

## 5.5 Other Critical Optimizations

In [ ]:
# ============================================
# AVOID UDFs - use built-in functions!
# ============================================

# BAD: Python UDF (slow - serialization overhead)
from pyspark.sql.functions import udf

@udf("double")
def bad_multiply(x):
    return x * 2

# GOOD: Built-in function (optimized, runs in JVM)
df.withColumn("doubled", F.col("value") * 2)

In [ ]:
# If you MUST use Python logic, use pandas_udf (vectorized)
from pyspark.sql.functions import pandas_udf
import pandas as pd

@pandas_udf("double")
def vectorized_multiply(s: pd.Series) -> pd.Series:
    return s * 2

df.withColumn("doubled", vectorized_multiply(F.col("value")))

In [ ]:
# ============================================
# CACHING
# ============================================

# Use when DataFrame is reused multiple times
df_cached = df.cache()  # Memory only

# persist() for more control
from pyspark import StorageLevel
df_persisted = df.persist(StorageLevel.MEMORY_AND_DISK)

# Don't forget to unpersist when done!
df_cached.unpersist()

In [ ]:
# ============================================
# FILTER EARLY
# ============================================

# GOOD: Filter before join (less data to shuffle)
df_filtered = df_big.filter(F.col("date") >= "2025-01-01")
df_joined = df_filtered.join(df_small, "key")

# BAD: Filter after join (unnecessary processing)
# df_joined = df_big.join(df_small, "key").filter(F.col("date") >= "2025-01-01")

In [ ]:
# ============================================
# REPARTITION vs COALESCE
# ============================================

# repartition() - full shuffle, use to INCREASE partitions or rebalance
df.repartition(100)  # Redistribute data across 100 partitions
df.repartition("customer_id")  # Partition by column

# coalesce() - no full shuffle, use to DECREASE partitions
df.coalesce(10)  # Reduce to 10 partitions (avoids shuffle)

## 5.6 Adaptive Query Execution (AQE)

Enabled by default in Databricks. Know what it does:
- Automatically coalesces small partitions
- Converts sort-merge joins to broadcast joins when appropriate
- Handles data skew

You don't need to configure it, but mention it shows you know modern Spark.

---
# 6. Unity Catalog

**Databricks' governance layer.** Virtually every new project uses it.

## 6.1 Three-Level Namespace

`catalog.schema.table` (e.g., `production.silver.meter_readings`)

In [ ]:
# Set default catalog and schema
spark.sql("USE CATALOG production")
spark.sql("USE SCHEMA silver")

# Now you can reference tables without full path
df = spark.read.table("meter_readings")  # = production.silver.meter_readings

In [ ]:
# Query across catalogs
df_prod = spark.read.table("production.silver.meter_readings")
df_dev = spark.read.table("development.bronze.raw_data")

## 6.2 Key Unity Catalog Concepts

| Concept | Description |
|---------|-------------|
| **Catalogs** | Top-level container (e.g., production, development) |
| **Schemas** | Container for tables within a catalog (like databases) |
| **Managed tables** | Data stored in Unity Catalog-managed storage |
| **External tables** | Data stored in your own cloud storage |
| **Volumes** | For file storage (unstructured data) |
| **Data lineage** | Automatic tracking of data flow |
| **Row/Column security** | Fine-grained access control |

In [ ]:
# Create catalog and schema
spark.sql("CREATE CATALOG IF NOT EXISTS my_project")
spark.sql("CREATE SCHEMA IF NOT EXISTS my_project.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS my_project.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS my_project.gold")

---
# 7. Data Quality

Interviewers love asking how you ensure data quality.

## 7.1 Manual Validation with PySpark

In [ ]:
# Add quality flags
df_with_quality = df.withColumn(
    "is_valid_reading",
    (F.col("reading_value") > 0) & 
    (F.col("reading_value") < 100000) &
    (F.col("reading_value").isNotNull())
)

# Split into valid and quarantine
valid_df = df_with_quality.filter(F.col("is_valid_reading") == True)
quarantine_df = df_with_quality.filter(F.col("is_valid_reading") == False)

# Write separately
valid_df.write.format("delta").mode("append").saveAsTable("silver.meter_readings")
quarantine_df.write.format("delta").mode("append").saveAsTable("quarantine.meter_readings")

In [ ]:
# Multiple quality checks
df_validated = df \
    .withColumn("check_not_null", F.col("reading_value").isNotNull()) \
    .withColumn("check_positive", F.col("reading_value") > 0) \
    .withColumn("check_reasonable", F.col("reading_value") < 100000) \
    .withColumn("is_valid", 
        F.col("check_not_null") & F.col("check_positive") & F.col("check_reasonable")
    )

## 7.2 Delta Live Tables (DLT) Expectations

DLT has built-in data quality. Know this exists even if you haven't used it deeply.

In [ ]:
# This is DLT syntax (runs in DLT pipelines, not regular notebooks)

import dlt

@dlt.table
@dlt.expect("valid_reading", "reading_value > 0 AND reading_value < 100000")
def silver_meter_readings():
    return spark.read.table("bronze.meter_readings")

# expect - warn but keep bad records
# expect_or_drop - drop bad records
# expect_or_fail - fail the pipeline

---
# 8. Common Interview Scenarios

These are practical questions you'll face in interviews.

## 8.1 Handling Slowly Changing Dimensions (SCD)

In [ ]:
# SCD Type 1: Overwrite (no history)
# Use MERGE with whenMatchedUpdateAll

target = DeltaTable.forName(spark, "dim.customers")

target.alias("t").merge(
    source_df.alias("s"),
    "t.customer_id = s.customer_id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

In [ ]:
# SCD Type 2: Keep history with start/end dates
# More complex - need to:
# 1. Close out old records (set end_date)
# 2. Insert new records with current start_date

target = DeltaTable.forName(spark, "dim.customers_scd2")

# Mark existing records as closed
target.alias("t").merge(
    source_df.alias("s"),
    "t.customer_id = s.customer_id AND t.is_current = true"
).whenMatchedUpdate(
    condition="t.name != s.name OR t.email != s.email",  # Only if changed
    set={
        "is_current": "false",
        "end_date": "current_timestamp()"
    }
).execute()

# Insert new current records
new_records = source_df \
    .withColumn("start_date", F.current_timestamp()) \
    .withColumn("end_date", F.lit(None).cast("timestamp")) \
    .withColumn("is_current", F.lit(True))

new_records.write.format("delta").mode("append").saveAsTable("dim.customers_scd2")

## 8.2 Handling Duplicate Data

In [ ]:
# Method 1: Simple dropDuplicates
df_deduped = df.dropDuplicates(["meter_id", "reading_timestamp"])

In [ ]:
# Method 2: Window function with row_number (more control)
# Keep the most recently ingested record for each key

window = Window.partitionBy("meter_id", "reading_timestamp") \
               .orderBy(F.col("_ingested_at").desc())

df_deduped = df \
    .withColumn("rn", F.row_number().over(window)) \
    .filter(F.col("rn") == 1) \
    .drop("rn")

## 8.3 Debugging Slow Jobs

**"Your job is running slow. How do you debug?"**

1. **Check Spark UI:**
   - Look at stages tab - which stage is taking longest?
   - Check for **skew** - one partition much larger than others
   - Check shuffle read/write sizes - high shuffle = expensive
   - Look for spill to disk - memory pressure

2. **Common fixes:**
   - **Skew:** Salting keys, repartition
   - **Large shuffle:** Broadcast small tables
   - **Many small files:** OPTIMIZE, coalesce before write
   - **Slow filters:** Add Z-Order on filter columns
   - **UDFs:** Replace with built-in functions

In [ ]:
# Explain plan to understand query execution
df.explain(mode="formatted")

## 8.4 Testing PySpark Code

In [ ]:
# Create small test DataFrames inline
def test_deduplication():
    # Arrange
    test_data = spark.createDataFrame([
        ("M001", "2025-01-01", 100.0),
        ("M001", "2025-01-01", 100.0),  # Duplicate
        ("M001", "2025-01-02", 150.0),
    ], ["meter_id", "date", "value"])
    
    # Act
    result = test_data.dropDuplicates(["meter_id", "date"])
    
    # Assert
    assert result.count() == 2, f"Expected 2 rows, got {result.count()}"
    print("Test passed!")

test_deduplication()

In [ ]:
# Compare DataFrames
def assert_dataframes_equal(df1, df2, order_by_cols):
    """Compare two DataFrames for equality."""
    df1_sorted = df1.orderBy(order_by_cols).collect()
    df2_sorted = df2.orderBy(order_by_cols).collect()
    assert df1_sorted == df2_sorted, "DataFrames are not equal"
    print("DataFrames match!")

---
# 9. Quick Reference Cheat Sheet

## Must-Know Functions

In [ ]:
# String functions
F.lower("col"), F.upper("col"), F.trim("col")
F.concat("col1", "col2"), F.concat_ws("-", "col1", "col2")
F.substring("col", 1, 5), F.length("col")
F.regexp_replace("col", "pattern", "replacement")
F.split("col", ","), F.explode("array_col")

# Date/Time functions
F.current_date(), F.current_timestamp()
F.to_date("col"), F.to_timestamp("col")
F.date_add("col", 7), F.date_sub("col", 7)
F.datediff("end", "start"), F.months_between("end", "start")
F.year("col"), F.month("col"), F.dayofmonth("col")
F.date_trunc("month", "col"), F.date_format("col", "yyyy-MM-dd")

# Null handling
F.coalesce("col1", "col2"), F.when(condition, value).otherwise(default)
F.col("x").isNull(), F.col("x").isNotNull()
F.na.fill({"col": "default"}), F.na.drop()

# Aggregations
F.count("*"), F.countDistinct("col")
F.sum("col"), F.avg("col"), F.min("col"), F.max("col")
F.collect_list("col"), F.collect_set("col")
F.first("col"), F.last("col")

# Window functions
F.row_number().over(window)
F.rank().over(window), F.dense_rank().over(window)
F.lag("col", 1).over(window), F.lead("col", 1).over(window)
F.sum("col").over(window)  # Running total

## Interview Answer Templates

| Question | Your Answer |
|----------|-------------|
| How do you handle late-arriving data? | "MERGE/upsert pattern - match on business key, update if exists, insert if new" |
| How do you roll back a bad write? | "RESTORE TABLE TO VERSION AS OF n - Delta's time travel makes this trivial" |
| Slow job - how to debug? | "Spark UI - check stages, look for skew, shuffle size, spill to disk. Then: broadcast, Z-Order, filter early" |
| How ensure data quality? | "Validation before write, quarantine bad records, use DLT expectations in production" |
| Partitioning strategy? | "Low-cardinality columns only (year, month, region). High-cardinality = small file problem" |

---
# 10. Practice Exercises

Complete these to solidify your understanding.

In [ ]:
# Exercise 1: Create a Bronze → Silver pipeline
# Given raw sales data, create a clean Silver table with:
# - Nulls removed
# - Dates properly typed
# - Duplicates removed (keep latest by ingestion time)
# - Add a 'sales_amount_usd' column (convert from cents to dollars)

# YOUR CODE HERE

In [ ]:
# Exercise 2: Write a MERGE statement
# Upsert customer data: update existing customers, insert new ones
# Track 'updated_at' timestamp for changed records

# YOUR CODE HERE

In [ ]:
# Exercise 3: Window function challenge
# For each customer, calculate:
# - Running total of purchases
# - Rank by purchase amount within each month
# - Days since last purchase

# YOUR CODE HERE

In [ ]:
# Exercise 4: Performance optimization
# Given this slow query, identify and fix performance issues:
"""
df_big = spark.read.table("fact.transactions")  # 100M rows
df_small = spark.read.table("dim.products")  # 10K rows

result = df_big \
    .join(df_small, "product_id") \
    .filter(F.col("transaction_date") >= "2025-01-01") \
    .groupBy("category").sum("amount")
"""

# YOUR OPTIMIZED CODE HERE

---
## Good luck with your interview!

**Key tips:**
1. Always explain your reasoning, not just the code
2. Mention trade-offs (e.g., "This approach is simpler but less scalable")
3. Reference Databricks-specific features (Delta, Unity Catalog, Auto Loader)
4. Talk about monitoring and debugging (Spark UI, EXPLAIN)
5. Show you think about data quality and governance